In [ ]:
import os

os.chdir("../../../")


import numpy as np
import matplotlib.pyplot as plt
from plot_serializer.matplotlib.serializer import MatplotlibSerializer

from src.postprocessing.get_quantities_along_path import process_pressure_branch_and_paths
from src.postprocessing.utils_plots import create_standard_list, save_data_raw

plt.style.use("FST_bw.mplstyle")

pt = 1./72.27
diss_tex_width = 390.0*pt

In [ ]:
# CONTROL STRATEGY NAMES:
CAV = "KVS"
VAVCPC = "KONST/ZENTRAL"
VAVVPC = "VAR/ZENTRAL"
DFCPC = "KONST/VERTEILT"
ONLYDF = "OHNE/VERTEILT"
ODSCC = "OPT"

control_strategies_names_ger = [CAV, VAVCPC, VAVVPC, DFCPC, ONLYDF, ODSCC]

control_strategies_names_ger_twocolumn = ["KVS", "KONST/\ZENTRAL", "VAR/\ZENTRAL", "KONST/\nVERTEILT", "OHNE/\nVERTEILT", "OPT"]

control_strategies_names_opt = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF"]

out_directory = "plots/change_along_path/"

created_with_file = "analyse_quantitiy_change_along_path.ipynb"

markerstyle = ["o","D", "s","v","^","P"]

## Compare all control strategies 

In [ ]:
general_information = {"author": {"name": "Julius H.P. Breuer", "ORCID": "0000-0002-6226-7208"}, "type": "PhD thesis, Technische Universität Darmstadt, Chair of Fluid Systems", "status": "in preparation", "title": "Algorithmische Systemplanung raumlufttechnischer Anlagen", "figure_created_with": "analyse_change_along_path.ipynb"}

def save_data(fig, serializer, outname, filedata, caption):
    save_data_raw(fig, serializer, out_directory, outname, filedata, caption, general_information=general_information, id_result_subfolder = "id_results/")

In [ ]:
def get_filenames_optimisation_type(folder_path, cs_names, optimisation_type="Topology"):
    paths = [folder_path + cs + "/" for cs in cs_names]

    standard_dict_list = create_standard_list(paths, optimisation_type, False)

    files = [folder_path + cs + "/" + str(x["filename"][0]) for cs, x in zip(cs_names, standard_dict_list)]
    return files

save_plots = input("Save all plots? y/n")
save_plots = True if save_plots.lower() == "y" else False

In [ ]:
s = 4

folder_path = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/standard_case/"


files = get_filenames_optimisation_type(folder_path, control_strategies_names_opt)

pressure_branch, root_room_paths, soundpower_branch = {}, {}, {}
for idx, file in enumerate(files):
    print(file)
    if file.split("/")[5] == "CAV":
        si = 1
        s_acoustic = 1
    else:
        si = s
        s_acoustic = 6
    dp_branch, path = process_pressure_branch_and_paths(file, "pressure")
    pressure_branch[control_strategies_names_opt[idx]] = dp_branch[si]
    root_room_paths[control_strategies_names_opt[idx]] = path

# markerstyle = ["o","D", "s","v","^","P"]

serializer = MatplotlibSerializer()
fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*4/9))


node = "2-1~6"

for idx, csi in enumerate(control_strategies_names_opt):
    dp_ = {key:value for key,value in pressure_branch[csi][node].items() }
    ax.plot(dp_.keys(), dp_.values(), label=csi, marker=markerstyle[idx])

n_elements = len(pressure_branch[control_strategies_names_opt[idx]][node])

ax.set_xlabel("ELEMENT")

ax.axhline(0)

ax.set_ylabel("STATISCHER DRUCK in Pa")
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

ax.legend(control_strategies_names_ger[:-1], loc='center left', bbox_to_anchor=(1, 0.5))

ax.set_xticklabels([])
fig.tight_layout()

for i in np.arange(0,n_elements):
    ax.axvline(i, color="grey", linewidth=0.2, zorder=-1)


if save_plots:
    outname = "pressure_distribution_GPZ"
    caption = r"Multifunktionsgebäude -- Topologieoptimierung\\Druckverlauf entlang des kritischen Strangs für alle Regelungsstrategien, dargestellt für einen Teillastfall. Zur vereinfachten Darstellung wurden aufeinanderfolgende Kanalabschnitte sowie festverbaute Element zusammengefasst. Ausgegraute Elemente wurden in keiner Regelstrategie gekauft. Bei den zentralen Strategien werden keine verteilten Ventilatoren eingesetzt, bei den verteilten Strategien keine VSR."
    filedata = [file.split("/")[-1][:-3] for file in files]
    save_data(fig, serializer, outname, filedata, caption)

## Compare acoustics of one solution

In [ ]:
s = 6

folder_path = "results/GPZ/combined/real/duct_dimensions_limited_by_real/"

cs_names = ["VAV-VPC"]

files = get_filenames_optimisation_type(folder_path, cs_names, "Configuration")


pressure_branch, root_room_paths, soundpower_branch = {}, {}, {}

file = files[-1]
cs = cs_names[-1]

if cs == "CAV":
    si = 1
    s_acoustic = 1
else:
    si = s
    s_acoustic = 6

spl_branch, _ = process_pressure_branch_and_paths(file, "sound_power_level")
soundpower_branch = spl_branch[s_acoustic]

node = "2-1~6"

n_elements = int(len(soundpower_branch[node])/8)


markerstyle = ["o","D", "s","v","^","P","x","*", "p", "h", "H"]

f_m = [63, 125, 250, 500, 1000, 2000, 4000, 8000]

serializer = MatplotlibSerializer()
fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*4/9))

greyscale_colors = np.linspace(0,1,9)[:-1]

for fi in range(1,9):
    dp_ = {key[0]:value for key,value in soundpower_branch[node].items() if key[1] == fi}
    ax.plot(dp_.keys(), dp_.values(), label=f"{f_m[fi-1]} Hz", marker=markerstyle[fi], color=str(greyscale_colors[fi-1]))

ax.set_xlabel("ELEMENT")

ax.set_ylim(bottom=0)

ax.set_ylabel("OKTAV-SCHALL-\nLEISTUNGSPEGEL in dB")
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

ax.set_xticklabels([])

for i in np.arange(0,n_elements):
    ax.axvline(i, color="grey", linewidth=0.2, zorder=-1)


fig.tight_layout()

if save_plots:
    outname = "spl_distribution_GPZ_" + cs
    caption = r"Multifunktionsgebäude -- Konfigurationsoptimierung\\Änderungen des Oktavband-Schallleistungspegels für einen Strang des Netzwerks im Maximallastfall. Die Symbole auf der $x$-Achse kennzeichnen das jeweils Pegeländerungen verursachende Element. Der zentrale Schalldämpfer wird nicht gekauft; der Schalldämpfer vor dem Raum reicht aus, um zulässige Schalldruckpegel nicht zu überschreiten."
    filedata = file.split("/")[-1][:-3]
    save_data(fig, serializer, outname, filedata, caption)

In [ ]:
from collections import defaultdict
import numpy as np

def pegel_add(*L):
    return 10 * np.log10(np.sum([10 ** (0.1 * x) for x in L]))

def level_adder(spl_branch):
    grouped = defaultdict(list)

    for (variant, octave), level in soundpower_branch[node].items():
        if octave in range(1, 9):   # nur Oktaven 1 bis 8
            grouped[variant].append(level)

    result = {
        variant: pegel_add(*levels)
        for variant, levels in grouped.items()
    }
    return result


In [ ]:
s = 6

folder_path = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/standard_case/"

cs_names = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC"]

files = get_filenames_optimisation_type(folder_path, cs_names, "Configuration")

nodes = ["1-2-1~6", "2-1~6"]

pressure_branch, root_room_paths, soundpower_branch = {}, {}, {}

serializer = MatplotlibSerializer()
fig,ax = serializer.subplots(1, 2, figsize=(diss_tex_width,diss_tex_width*3/9), sharey=True)

markerstyle = ["o","D", "s","v","^","P","x","*", "p", "h", "H"]

f_m = [63, 125, 250, 500, 1000, 2000, 4000, 8000]

greyscale_colors = np.linspace(0,1,9)[:-1]

for jdx, node in enumerate(nodes):

    for idx, (cs, file) in enumerate(zip(cs_names, files)):
        if cs == "CAV":
            s_acoustic = 1
        else:
            s_acoustic = 6
        spl_branch, _ = process_pressure_branch_and_paths(file, "sound_power_level")
        soundpower_branch = spl_branch[s_acoustic]

        n_elements = int(len(soundpower_branch[node])/8)
        spl_values = level_adder(soundpower_branch[node])
        ax[jdx].plot(spl_values.keys(), spl_values.values(), label=cs, marker=markerstyle[idx], markersize=3)
    for i in np.arange(0,n_elements):
        ax[jdx].axvline(i, color="grey", linewidth=0.2, zorder=-1)
    ax[jdx].set_xticklabels([])
    ax[jdx].set_xlabel("ELEMENT", fontsize=8)

ax[1].legend(control_strategies_names_ger[:-1], loc='center left', bbox_to_anchor=(1, 0.5), fontsize=7)
ax[1].set_ylim(bottom=0)
ax[0].set_ylabel("SCHALLLEISTUNGS-\nPEGEL in dB")

fig.tight_layout()

if save_plots:
    outname = "spl_distribution_all_cs_two_examples_GPZ"
    filedata = file.split("/")[-1][:-3]
    save_data(fig, serializer, outname, filedata, caption)

In [ ]:
from matplotlib.font_manager import FontProperties
arial_font = FontProperties(family="Arial", style="italic")


folder_path = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/standard_case/"

cs_names = ["VAV-CPC"]

files = get_filenames_optimisation_type(folder_path, cs_names, "Configuration")


pressure_branch, root_room_paths, soundpower_branch = {}, {}, {}

file = files[-1]
cs = cs_names[-1]


spl_branch, _ = process_pressure_branch_and_paths(file, "sound_power_level")
soundpower_branch = spl_branch[6]

node = "1-1~5"

n_elements = int(len(soundpower_branch[node])/8)


markerstyle = ["o","D", "s","v","^","P","x","*", "p", "h", "H"]

f_m = [63, 125, 250, 500, 1000, 2000, 4000, 8000]

serializer = MatplotlibSerializer()
fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*4/9))


greyscale_colors = np.linspace(0,1,9)[:-1]

for fi in range(1,9):
    dp_ = {key[0]:value for key,value in soundpower_branch[node].items() if key[1] == fi}
    if fi == 8:
        ax.plot(dp_.keys(), dp_.values(), label="Minimallastfall", marker=markerstyle[0], color="k")
    # else:
    #     ax.plot(dp_.keys(), dp_.values(),  marker=markerstyle[0], color="grey", zorder=-1)

ax.set_xlabel("ELEMENT")

ax.set_ylim(bottom=0)

ax.set_ylabel("OKTAV-SCHALL-\nLEISTUNGSPEGEL in dB")
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))

ax.set_xticklabels([])


for i in np.arange(0,n_elements):
    ax.axvline(i, color="grey", linewidth=0.2, zorder=-1)

soundpower_branch = spl_branch[1]

node = "1-1~5"

n_elements = int(len(soundpower_branch[node])/8)


for fi in range(1,9):
    dp_ = {key[0]:value for key,value in soundpower_branch[node].items() if key[1] == fi}
    if fi == 8:
        ax.plot(dp_.keys(), dp_.values(), label="Minimallastfall", marker=markerstyle[1], color="darkgrey")
    # else:
    #     ax.plot(dp_.keys(), dp_.values(),  marker=markerstyle[1], color="lightgrey", zorder=-1)


ax.set_ylim(bottom=0)

ax.set_ylabel("OKTAV-SCHALL-\nLEISTUNGSPEGEL in dB")
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), prop=arial_font)

ax.set_xticklabels([])

for i in np.arange(0,n_elements):
    ax.axvline(i, color="grey", linewidth=0.2, zorder=-1)

fig.tight_layout()

if save_plots:
    outname = "spl_distribution_vavcpc_explanation_8000Hz_Room_1_1"
    filedata = file.split("/")[-1][:-3]
    save_data(fig, serializer, outname, filedata, caption)